# Generating a weighted lightcone of Diffsky galaxies
This notebook demonstrates how to generate a lightcone of diffsky galaxies with SEDs, star formation histories, dust, and other properties.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Download stellar population synthesis data
The very first thing we will do in this notebook is download some LSST-like transmission curves, and also some SEDs of simple stellar populations (SSPs) used in the SED modeling. You can skip this step of you have already cached previously-stored SSP data as described in the [DSPS Quickstart Guide](https://dsps.readthedocs.io/en/latest/quickstart.html).

Before generating these docs, these data were already been downloaded, and so we will just load them directly. See the DSPS docs for download instructions.

#### The `load_ssp_templates` function in Diffsky loads an hdf5 file and packs it into a namedtuple with the expected field names.

> **Note:** The original demo uses a sparse version of these SSP templates to reduce memory use when building the documentation. Here we use the full-resolution templates.

In [ ]:
from diffsky.data_loaders import load_ssp_templates
ssp_data = load_ssp_templates(
    drn="dsps_drn",
    bn="fsps_v0.4.7_mist_c3k_a_kroupa_wNE_logGasU-2.0_logGasZ0.0.sparse.h5")

print(ssp_data._fields)

#### The `load_transmission_curve` function in DSPS loads an hdf5 file and packs it into a namedtuple with the expected field names.¶

You can use any transmission curves to compute Diffsky photometry. You just need to create a namedtuple storing ‘wave’, ‘transmission’ of each transmission curve.

In [ ]:
from dsps.data_loaders import load_transmission_curve
bn_base_pat = "lsst_{}*"
filter_names = ('u', 'g', 'r', 'i', 'z', 'y')
bn_pat_list = [bn_base_pat.format(x) for x in filter_names]
tcurves = [load_transmission_curve(bn_pat=pat, drn="dsps_drn/filters") for pat in bn_pat_list]

print(tcurves[0]._fields)

## Generate the halo lightcone and additional data needed to model/compute SEDs

First specify halo lightcone specs:
* `num_halos`: the number of grid points in mass and redshift for the host halos in the lightcone. Each host halo on the grid is populated with a small number (typically 5) of lower-mass subhalos.
* `z_min, z_max`: the redshift range
* `lgmp_min` should be small enough to resolve the faintest galaxies in your population (values of 10-11 are typical for cosmological surveys).
* `lgmp_max` is the largest halo mass in the population. In most cases, this should typically be set to 15 or 16.
* `sky_area_degsq` is the sky area in square degrees.

In [ ]:
from diffsky.lc_generators.lc_data_phot import weighted_lc_data_phot

num_halos = 500
z_min, z_max = 0.1, 2.0
lgmp_min, lgmp_max = 10.5, 15.0
sky_area_degsq = 10.0

#### Get a random number seed from JAX

In [ ]:
from jax import random as jran
ran_key = jran.key(0)

#### Define a redshift table used for photometry interpolation

In [ ]:
n_z_phot_table = 15
z_phot_table = np.linspace(z_min, z_max, n_z_phot_table)

#### Generate the lightcone of dark matter halos and additional SPS data

In [ ]:
halo_lc_data = (num_halos, z_min, z_max, lgmp_min, lgmp_max, sky_area_degsq)
phot_data = (ssp_data, tcurves, z_phot_table)

ran_key, lc_halo_key = jran.split(ran_key, 2)
args = (lc_halo_key, *halo_lc_data, *phot_data)

lc_data = weighted_lc_data_phot(*args)
print(lc_data._fields)

## Populate the lightcone with diffsky galaxies

Now we will pass the `lc_data` to the diffsky SED kernels to populate the halo lightcone with galaxies.

In [ ]:
from diffsky.sed_models.ou26.lc_phot import mc_lc_phot

mc_merge = 1  # selecting Monte Carlo merging
ran_key, sed_key = jran.split(ran_key, 2)
phot_info, __, __ = mc_lc_phot(sed_key, lc_data, mc_merge)
print(phot_info._fields)

In [ ]:
phot_info.ssp_weights.shape

## Choosing an alternative set of diffsky model parameters

There are several choices for diffsky parameters that are based on ongoing work calibrating Diffsky to the COSMOS and FENIKS-UDS datasets. Below we show how to check which parameters are available.

In [ ]:
from diffsky.param_utils import DIFFSKY_FIT_PARAMS
print(DIFFSKY_FIT_PARAMS.keys())

In [ ]:
cosmos_sdss_param_collection = DIFFSKY_FIT_PARAMS['cs261020']
feniks_sdss_hizels_param_collection = DIFFSKY_FIT_PARAMS['sdss_feniks_hizels_261013']

#### Generate photometry with alternative set of model parameters

In [ ]:
phot_info_feniks_sdss_hizels, _, _ = mc_lc_phot(
    sed_key, lc_data, mc_merge, 
    param_collection=feniks_sdss_hizels_param_collection)

## Understanding the _weighted_ aspect of a _weighted_ lightcone

#### The halo sample in redshift and halo mass

Instead of sampling halos in proportion to their abundance, the representative host halos uniformly cover the $(z, \log_{10}M_{\rm halo})$ plane. Each host carries a fixed number of subhalos at the same redshift. Abundance information is entirely captured in the weights.

In [ ]:
cen_mask = (lc_data.is_central==1)
fig, ax = plt.subplots(1, 1)
sc = ax.scatter(10**lc_data.logmp_obs[cen_mask], lc_data.z_obs[cen_mask], c=np.log10(lc_data.halo_weight[cen_mask]),s=3)
ax.set(xlabel=r'$M_{\rm halo}\ [M_{\odot}]$', xscale='log', ylabel='redshift')
fig.colorbar(sc, ax=ax, label=r'$\log \, ({\rm halo\_weight})$')
plt.show()

#### For the _weighted_ lightcone, each halo has multiplicity according to its abundance in the volume

This `halo_weight` column needs to be taken into account when predicting summary statistics from the weighted lightcone. This column stores how each grid point should be weighted in accord with the abundance of halos of the mass and redshift of the grid point.

In [ ]:
fig, ax = plt.subplots(1, 1)
__=ax.loglog()
__=ax.scatter(10**lc_data.logmp_obs[lc_data.is_central==1],
              lc_data.halo_weight[lc_data.is_central==1], s=1, label=r'${\rm centrals}$')
__=ax.scatter(10**lc_data.logmp_obs[lc_data.is_central==0],
              lc_data.halo_weight[lc_data.is_central==0], s=1, label=r'${\rm satellite}$')
leg = ax.legend(markerscale=4)
xlabel = ax.set_xlabel(r'$M_{\rm halo}\ [M_{\odot}]$')
ylabel = ax.set_ylabel(r'${\rm halo\_{weight}}$')

#### Calculate the halo mass function, accounting for halo weights

The unweighted version uniformly spans $\log_{10}M_{\rm halo}$. The weighted version has the expected Schechter-type shape of the HMF.

In [ ]:
fig, ax = plt.subplots(1, 1)
yscale = ax.set_yscale('log')
__=ax.hist(lc_data.logmp_obs, bins=50, alpha=0.3, label=r'${\rm unweighted}$')
__=ax.hist(lc_data.logmp_obs, bins=50, weights=lc_data.halo_weight, alpha=1, linewidth=2, histtype='step', label=r'${\rm weighted}$')
xlabel = ax.set_xlabel(r'$\log_{10}M_{\rm halo}/M_{\odot}$')
ylabel = ax.set_ylabel(r'$N_{\rm halos}$')
leg = ax.legend()

## Inspecting the galaxies

#### Visually inspect the diversity of SFHs

In [ ]:
fig, ax = plt.subplots(1, 1)
yscale = ax.set_yscale('log')
ylim = ax.set_ylim(8e-4, 1e3)
xscale = ax.set_xscale('log')
xlim = ax.set_xlim(1, 15)

n_plot = 5
for i in range(n_plot):
    __=ax.plot(lc_data.t_table, phot_info.sfh_table[i, :])

xlabel = ax.set_xlabel(r'${\rm cosmic\ time\ [Gyr]}$')
ylabel = ax.set_ylabel(r'${\rm SFR\ [M_{\odot}/yr]}$')

#### Visually inspect the sSFR PDF

In [ ]:
fig, ax = plt.subplots(1, 1)
xlim = ax.set_xlim(-15, -7)
yscale = ax.set_yscale('log')
ax.hist(phot_info.logssfr_obs, bins=70, alpha=0.3, density=True, label=r'${\rm unweighted}$')
ax.hist(phot_info.logssfr_obs, bins=70, alpha=1, weights=lc_data.halo_weight, density=True, histtype='step', linewidth=2, label=r'${\rm weighted}$')
xlabel = ax.set_xlabel(r'${\rm log_{10}(sSFR)}$')
ylim = ax.set_ylim(1e-3, 5e0)
leg = ax.legend()

#### Visually inspect star-forming sequence

Note: This is an unweighted scatter plot. To get the correct distribution one needs to have a binned 2D histogram or a contour plot accounting for the weights.

In [ ]:
fig, ax = plt.subplots(1, 1)
ylim = ax.set_ylim(-13.5, -7.5)
xlim = ax.set_xlim(7.5, 13)
__=ax.scatter(phot_info.logsm_obs, phot_info.logssfr_obs, s=1)
xlabel = ax.set_xlabel(r'${\rm log_{10}(M_{\star})}$')
ylabel = ax.set_ylabel(r'${\rm log_{10}(sSFR)}$')

#### Plot color-color diagram
Note that the plot below shows the PDF for all galaxies/halos in the lightcone, without accounting for the weights, and without selecting a particular galaxy sample of interest.

In [ ]:
fig, ax = plt.subplots(1, 1)

ri = phot_info.obs_mags[:, 2]-phot_info.obs_mags[:, 3]
iz = phot_info.obs_mags[:, 3]-phot_info.obs_mags[:, 4]
__=ax.scatter(iz, ri, s=1)
xlabel = ax.set_xlabel(r'${\rm i-z}$')
ylabel = ax.set_ylabel(r'${\rm r-i}$')

#### Plot color–redshift relation
Note that the plot below shows the PDF for all galaxies/halos in the lightcone, without accounting for the weights, and without selecting a particular galaxy sample of interest.

In [ ]:
fig, ax = plt.subplots(1, 1)

iz = phot_info.obs_mags[:, 3]-phot_info.obs_mags[:, 4]
__=ax.scatter(lc_data.z_obs, iz, s=1)
xlabel = ax.set_xlabel(r'${\rm redshift}$')
ylabel = ax.set_ylabel(r'${\rm i-z}$')

## Weighted summary statistics

Every statistic must weight each galaxy by `halo_weight`.

#### Galaxy apparent magnitude distributions in redshift bins

In [ ]:
N_zbins = 4
N_magbins = 20
z_edges = np.linspace(z_min, z_max, N_zbins+1)
mag_edges = np.linspace(16, 25, N_magbins+1)
colors = ['tab:blue','tab:green','tab:olive','tab:orange','tab:red','tab:brown'] # each for a different filter

weights = np.asarray(lc_data.halo_weight)
redshifts = np.asarray(lc_data.z_obs)
app_mags = np.asarray(phot_info.obs_mags)

fig, axs = plt.subplots(1,N_zbins, figsize=(2*N_zbins,2), sharex=True,sharey=True)

for zi in range(N_zbins):
    z_mask = (redshifts > z_edges[zi])&(redshifts <= z_edges[zi+1])
    app_mags_subset = app_mags[z_mask]
    weights_subset = weights[z_mask]
    for mi in range(len(filter_names)):
        axs[zi].hist(app_mags_subset[:,mi],bins=mag_edges, weights=weights_subset, color=colors[mi],histtype='step',label=filter_names[mi])
    axs[zi].set(xlabel='app mag',yscale='log', ylim=(1e0,1e6))
    axs[zi].set_title(f'{round(z_edges[zi],2)} < z < {round(z_edges[zi+1],2)}',fontsize=10)

axs[zi].legend()
axs[0].set_ylabel('counts')

#### Redshift distributions of magnitude-limited samples

In [ ]:
N_zbins = 20
mag_limits = (23.0, 24.0, 25.0)
limiting_band = 'i'

mag_index = filter_names.index(limiting_band)
z_edges = np.linspace(z_min, z_max, N_zbins+1)
dz = np.diff(z_edges)

weights = np.asarray(lc_data.halo_weight)
redshifts = np.asarray(lc_data.z_obs)
app_mags = np.asarray(phot_info.obs_mags)

fig, axs = plt.subplots(1, 2, figsize=(10, 4.5))
for m_lim in mag_limits:
    selection = app_mags[:,mag_index] < m_lim
    counts_w, _ = np.histogram(redshifts[selection], bins=z_edges, weights=weights[selection])
    counts_uw, _ = np.histogram(redshifts[selection], bins=z_edges)
    axs[0].stairs(counts_uw, z_edges, label=f'{limiting_band} < {m_lim}')
    axs[1].stairs(counts_w / dz / sky_area_degsq, z_edges, label=f'{limiting_band} < {m_lim}')

axs[0].set(title='without accounting for weights',ylabel='counts')
axs[1].set(title='correct weighted version',ylabel=r'$dN \, / \, dz \,\,\, [{\rm deg}^{-2}]$')
for ax in axs:
    ax.set_xlabel('redshift')
    ax.legend()
fig.tight_layout()

## Weighted versus Monte Carlo lightcones

`mc_lc_data_phot` generates a Monte Carlo version of a lightcone, where the halos are sampled to have the correct cosmological abundance according to halo mass function, and weights are all unity. The central and satellite counts from the weighted and Monte Carlo lightcones should be close to each other, and approach the same value for large sky area and large `num_halos` (in the weighted one).

In [ ]:
from diffsky.lc_generators.lc_data_phot import mc_lc_data_phot

z_min, z_max = 0.1, 3.0
sky_area_degsq = 10.0  # deg^2
lgmp_min = 11.5
lgmp_max = 15.0  # this is for the weighted lightcone only
num_halos = 500    # this is for the weighted lightcone only

ran_key, mc_lc_key, w_lc_key = jran.split(ran_key, 3)

# Monte Carlo lightcone
lc_data_mc = mc_lc_data_phot(
    mc_lc_key, z_min, z_max, lgmp_min, lgmp_min, sky_area_degsq, ssp_data, tcurves, z_phot_table
)

# Weighted lightcone
lc_data_w = weighted_lc_data_phot(
    w_lc_key, num_halos, z_min, z_max, lgmp_min, lgmp_max, sky_area_degsq, ssp_data, tcurves, z_phot_table
)

for name, lc in (("Monte Carlo", lc_data_mc), ("Weighted lightcone", lc_data_w)):
    cen_mask = (np.asarray(lc.is_central) == 1)
    sat_mask = ~cen_mask
    print(f"{name}: generated {len(lc.halo_weight)} (sub)halos")
    print(f"N_cen = {lc.halo_weight[cen_mask].sum():.1f}, N_sat = {lc.halo_weight[sat_mask].sum():.1f}")
    print("---")

#### Halo mass function and redshift distribution: Weighted versus Monte Carlo

In [ ]:
lgm_bins = np.linspace(lgmp_min, 14.5, 25)
z_bins = np.linspace(z_min, z_max, 20)

fig, axs = plt.subplots(1, 2, figsize=(8, 3))

axs[0].hist(lc_data_w.logmp_obs[lc_data_w.is_central==1], bins=lgm_bins, weights=lc_data_w.halo_weight[lc_data_w.is_central==1], label='Weighted', histtype='step', linewidth=2)
axs[1].hist(lc_data_w.z_obs, bins=z_bins, weights=lc_data_w.halo_weight, histtype='step', linewidth=2)

axs[0].hist(lc_data_mc.logmp_obs[lc_data_mc.is_central==1], bins=lgm_bins, label='Monte Carlo', alpha=0.3)
axs[1].hist(lc_data_mc.z_obs, bins=z_bins, alpha=0.3)

axs[0].set_yscale('log')
axs[0].set_xlabel(r'$\log M_{\rm halo} \,\, [M_{\odot}]$')
axs[0].set_ylabel(r'$N_{\rm host}$')
axs[1].set_xlabel(r'${\rm redshift}$')
axs[1].set_ylabel(r'$N_{\rm gal}\ ({\rm cen+sat})$')

axs[0].legend()
fig.tight_layout()

## Migrating from diffsky.experimental to the code in this notebook

For those used to the older version of this demo notebook, the key updates to note are as follows:
- `diffsky.experimental.lightcone_generators.weighted_lc_photdata` $\rightarrow$ `diffsky.lc_generators.lc_data_phot.weighted_lc_data_phot`
- `diffsky.experimental.mc_phot.mc_lc_phot` $\rightarrow$ `diffsky.sed_models.ou26.lc_phot.mc_lc_phot`